# V5.1 — 12M humanizer experiment

This notebook runs the 12M candidate in two stages: a balanced base phase, then the H1-matched expert-edit curriculum. V4.8 remains untouched. Calibration and sealed evaluation happen only after a candidate passes the development promotion comparison.

## 1. Setup
Run this once after choosing a GPU runtime.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/humanized-ai-likelihood-v51')
if not REPO.exists():
    !git clone --branch codex/v5-1-12m --single-branch https://github.com/bonbon1235312/googlecolab-humanize-detector.git /content/humanized-ai-likelihood-v51

%cd /content/humanized-ai-likelihood-v51/ml
!git fetch origin codex/v5-1-12m
!git switch codex/v5-1-12m
!git pull --ff-only
!pip -q install -e .
!nvidia-smi

## 2. Check inputs
These are deliberately separate: the timing run is disposable, V5.1 has a new output folder, and V4.8 is read-only.

In [ ]:
CONTROL_DATA_DIR = '/content/drive/MyDrive/v4-data/control-v1'
V48_ARTIFACTS = '/content/drive/MyDrive/v4-artifacts/v4-8/masked_mean_base'
TIMING_ARTIFACTS = '/content/drive/MyDrive/v5-artifacts/v5-1-12m-timing'
V51_ARTIFACTS = '/content/drive/MyDrive/v5-artifacts/v5-1-12m-final'

for path in (Path(CONTROL_DATA_DIR) / 'train.jsonl', Path(CONTROL_DATA_DIR) / 'development.jsonl', Path(V48_ARTIFACTS) / 'model.pt'):
    assert path.exists(), f'Missing required input: {path}'
print('Inputs found. V4.8 will not be modified.')

## 3. Timing gate
This executes exactly one base epoch and estimates the complete 6-base + 4-curriculum job. Do not run the full base phase until its estimate fits the available runtime.

In [ ]:
!python -u -m humanized_detector.v5_1_train --data-dir $CONTROL_DATA_DIR --artifacts-dir $TIMING_ARTIFACTS --stage base --base-epochs 6 --curriculum-epochs 4 --batch-size 64 --lr 3e-5 --weight-decay 0.01 --label-smoothing 0.02 --warmup-fraction 0.05 --grad-clip-norm 1.0 --timing-only

## 4. Balanced base training
Run only after the timing estimate is acceptable. This starts a fresh 12M model and selects its best development checkpoint.

In [ ]:
!python -u -m humanized_detector.v5_1_train --data-dir $CONTROL_DATA_DIR --artifacts-dir $V51_ARTIFACTS --stage base --base-epochs 6 --curriculum-epochs 4 --batch-size 64 --lr 3e-5 --weight-decay 0.01 --label-smoothing 0.02 --warmup-fraction 0.05 --grad-clip-norm 1.0

## 5. H1-matched curriculum
This loads the selected balanced-base checkpoint and applies the 25/50/25 → 40/35/25 → 50/30/20 Beemo-positive curriculum.

In [ ]:
!python -u -m humanized_detector.v5_1_train --data-dir $CONTROL_DATA_DIR --artifacts-dir $V51_ARTIFACTS --stage curriculum --base-epochs 6 --curriculum-epochs 4 --batch-size 64 --lr 3e-5 --weight-decay 0.01 --label-smoothing 0.02 --warmup-fraction 0.05 --grad-clip-norm 1.0

## 6. Promotion comparison
Both candidates are compared with V4.8 on development only. The bootstrap resamples complete lineages, requires credible macro improvement, and blocks meaningful subtype or low-FPR regressions.

In [ ]:
!python -u -m humanized_detector.v5_1_compare --data-dir $CONTROL_DATA_DIR --baseline-artifacts $V48_ARTIFACTS --candidate-artifacts $V51_ARTIFACTS/base --output-dir $V51_ARTIFACTS/base_comparison --iterations 2000
!python -u -m humanized_detector.v5_1_compare --data-dir $CONTROL_DATA_DIR --baseline-artifacts $V48_ARTIFACTS --candidate-artifacts $V51_ARTIFACTS/curriculum --output-dir $V51_ARTIFACTS/curriculum_comparison --iterations 2000

## 7. Calibrate only a promoted candidate
Set `SELECTED_ARTIFACTS` only if its `promotion_decision.json` contains `"promote": true`. If neither candidate passes, keep V4.8.

In [ ]:
SELECTED_ARTIFACTS = f'{V51_ARTIFACTS}/base'  # Change to /curriculum only if that candidate passes.
!python -u -m humanized_detector.v5_1_calibrate --data-dir $CONTROL_DATA_DIR --artifacts-dir $SELECTED_ARTIFACTS